In [1]:
pip install datasets

In [13]:
import pandas as pd
import re
import json

In [17]:
with open("/content/breast_cancer_finetune_data.json", "r") as file:
  content = json.load(file)

In [18]:
len(content)

83

In [19]:
df1 = pd.read_csv("/content/train.csv")

In [20]:
df2 = pd.read_csv("/content/medquad.csv")

In [7]:
df1.head()

,qtype,Question,Answer
0,susceptibility,Who is at risk for Lymphocytic Choriomeningiti...,LCMV infections can occur after exposure to fr...
1,symptoms,What are the symptoms of Lymphocytic Choriomen...,LCMV is most commonly recognized as causing ne...
2,susceptibility,Who is at risk for Lymphocytic Choriomeningiti...,Individuals of all ages who come into contact ...
3,exams and tests,How to diagnose Lymphocytic Choriomeningitis (...,"During the first phase of the disease, the mos..."
4,treatment,What are the treatments for Lymphocytic Chorio...,"Aseptic meningitis, encephalitis, or meningoen..."


In [8]:
df2.head()

,question,answer,source,focus_area
0,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma
1,What causes Glaucoma ?,"Nearly 2.7 million people have glaucoma, a lea...",NIHSeniorHealth,Glaucoma
2,What are the symptoms of Glaucoma ?,Symptoms of Glaucoma Glaucoma can develop in ...,NIHSeniorHealth,Glaucoma
3,What are the treatments for Glaucoma ?,"Although open-angle glaucoma cannot be cured, ...",NIHSeniorHealth,Glaucoma
4,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma


In [9]:
df1.columns

Index(['qtype', 'Question', 'Answer'], dtype='object')

In [10]:
df1 = df1[['Question', 'Answer']]

In [29]:
df1.rename(
    columns={
        "Question":"question",
        "Answer": "answer"
    },
    inplace=True
)

In [30]:
df1.head()

,question,answer
0,Who is at risk for Lymphocytic Choriomeningiti...,LCMV infections can occur after exposure to fr...
1,What are the symptoms of Lymphocytic Choriomen...,LCMV is most commonly recognized as causing ne...
2,Who is at risk for Lymphocytic Choriomeningiti...,Individuals of all ages who come into contact ...
3,How to diagnose Lymphocytic Choriomeningitis (...,"During the first phase of the disease, the mos..."
4,What are the treatments for Lymphocytic Chorio...,"Aseptic meningitis, encephalitis, or meningoen..."


In [19]:
df1.shape

(16407, 2)

In [13]:
df2.columns

Index(['question', 'answer', 'source', 'focus_area'], dtype='object')

In [14]:
df2 = df2[['question', 'answer']]

In [31]:
df2.head()

,question,answer
0,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...
1,What causes Glaucoma ?,"Nearly 2.7 million people have glaucoma, a lea..."
2,What are the symptoms of Glaucoma ?,Symptoms of Glaucoma Glaucoma can develop in ...
3,What are the treatments for Glaucoma ?,"Although open-angle glaucoma cannot be cured, ..."
4,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...


In [20]:
df2.shape

(16412, 2)

In [32]:
df_merged = pd.concat([df1, df2], ignore_index=True)

In [33]:
df_merged.head()

,question,answer
0,Who is at risk for Lymphocytic Choriomeningiti...,LCMV infections can occur after exposure to fr...
1,What are the symptoms of Lymphocytic Choriomen...,LCMV is most commonly recognized as causing ne...
2,Who is at risk for Lymphocytic Choriomeningiti...,Individuals of all ages who come into contact ...
3,How to diagnose Lymphocytic Choriomeningitis (...,"During the first phase of the disease, the mos..."
4,What are the treatments for Lymphocytic Chorio...,"Aseptic meningitis, encephalitis, or meningoen..."


In [37]:
df_merged.dropna(inplace=True)

In [39]:
df_merged['question'] = df_merged['question'].astype(str).str.strip()
df_merged['answer'] = df_merged['answer'].astype(str).str.strip()


In [40]:
df_merged["q_normalized"] = (
    df_merged["question"].str.lower().str.replace(r"[^\w\s]", "", regex=True).str.strip()
)

In [41]:
df_merged = df_merged.drop_duplicates(subset="q_normalized").drop(columns=['q_normalized'])

In [42]:
df_merged.shape

(14336, 2)

In [43]:
df_merged.head()

,question,answer
0,Who is at risk for Lymphocytic Choriomeningiti...,LCMV infections can occur after exposure to fr...
1,What are the symptoms of Lymphocytic Choriomen...,LCMV is most commonly recognized as causing ne...
3,How to diagnose Lymphocytic Choriomeningitis (...,"During the first phase of the disease, the mos..."
4,What are the treatments for Lymphocytic Chorio...,"Aseptic meningitis, encephalitis, or meningoen..."
5,How to prevent Lymphocytic Choriomeningitis (L...,LCMV infection can be prevented by avoiding co...


In [44]:
BREAST_CANCER_KEYWORDS = [
    "breast cancer", "breast tumor", "breast tumour", "mammogram", "mammography",
    "brca", "mastectomy", "lumpectomy", "breast biopsy", "breast lump",
    "ductal carcinoma", "lobular carcinoma", "her2", "triple negative",
    "breast self-exam", "breast screening",
]

In [45]:
pattern = "|".join(re.escape(k) for k in BREAST_CANCER_KEYWORDS)

In [47]:
mask = (
    df_merged["question"].str.contains(pattern, case=False, na=False)
    | df_merged["answer"].str.contains(pattern, case=False, na=False)
)

In [48]:
breast_cancer_subset = df_merged[mask].reset_index(drop=True)

In [49]:
print(f"Breast-cancer-relevant rows found: {len(breast_cancer_subset)}")

Breast-cancer-relevant rows found: 83


In [50]:
df_merged.to_csv("merged_medical_qa_full.csv", index=False)

In [51]:
breast_cancer_subset.to_csv("breast_cancer_qa_subset.csv", index=False)

In [53]:
instruction_pairs = [
    {
        "instruction": row["question"],
        "input": "",
        "output": row["answer"],
    }
    for _, row in breast_cancer_subset.iterrows()
]

In [55]:
with open("breast_cancer_finetune_data.json", "w") as f:
    json.dump(instruction_pairs, f, indent=2)

In [58]:
print(f"Saved {len(instruction_pairs)} instruction pairs to breast_cancer_finetune_data.json")
print("NOTE: If this count is < 500, you'll need to supplement with your own")
print("curated NCI/WHO/BreastCancer.org pairs to hit a reasonable fine-tuning size.")

Saved 83 instruction pairs to breast_cancer_finetune_data.json
NOTE: If this count is < 500, you'll need to supplement with your own
curated NCI/WHO/BreastCancer.org pairs to hit a reasonable fine-tuning size.


In [21]:
with open("/content/breast_cancer_finetune_data.json", "r") as file:
  content = json.load(file)

In [22]:
len(content)

83

In [24]:
content[0]

{'instruction': 'What is (are) Paraneoplastic Syndromes ?',
 'input': '',
 'output': 'Paraneoplastic syndromes are a group of rare disorders that are triggered by an abnormal immune system response to a cancerous tumor known as a "neoplasm." Paraneoplastic syndromes are thought to happen when cancer-fighting antibodies or white blood cells (known as T cells) mistakenly attack normal cells in the nervous system. These disorders typically affect middle-aged to older people and are most common in individuals with lung, ovarian, lymphatic, or breast cancer. Neurologic symptoms generally develop over a period of days to weeks and usually occur prior to the tumor being discovered. These symptoms may include difficulty in walking or swallowing, loss of muscle tone, loss of fine motor coordination, slurred speech, memory loss, vision problems, sleep disturbances, dementia, seizures, sensory loss in the limbs, and vertigo or dizziness. Paraneoplastic syndromes include Lambert-Eaton myasthenic s

In [11]:
a = [
  {"instruction": "What exactly is breast cancer?", "input": "", "output": "Breast cancer happens when cells in the breast start growing out of control and form a mass called a tumor, which can sometimes spread to other parts of the body. It can start in one or both breasts and mostly affects people aged 45 and older, though it's rare in children and in men. If you're worried about a specific symptom or diagnosis, please talk to your doctor \u2014 they can give you guidance specific to your situation."},
  {"instruction": "My doctor said I have 'ductal' breast cancer. What does that mean?", "input": "", "output": "Ductal cancers are cancers that start in the milk ducts, the thin tubes that carry milk to the nipple \u2014 and it's actually the most common type of breast cancer. Ductal cancer can be either 'in situ' (contained, hasn't spread) or invasive (has spread into surrounding tissue). Your doctor can tell you exactly which type you have and what it means for your treatment plan."},
  {"instruction": "What's the difference between ductal and lobular breast cancer?", "input": "", "output": "Ductal cancer starts in the milk ducts and is the most common type, while lobular cancer starts in the lobules, the glands that make milk. Lobular cancer tends to grow more slowly, is less common, and is more likely to show up in both breasts compared to ductal cancer. Your care team can explain which type applies to you and how that affects your treatment options."},
  {"instruction": "I was just told I have inflammatory breast cancer. Is that as scary as it sounds?", "input": "", "output": "It's understandable to feel scared \u2014 inflammatory breast cancer is a rare and more aggressive form where cancer cells block lymph vessels in the skin, often making the breast look swollen, red, or 'inflamed.' It does carry a higher risk of recurrence than some other types, which is exactly why your care team will likely move quickly on a treatment plan. Please lean on your doctor and care team for a plan tailored to you \u2014 you don't have to process this alone."},
  {"instruction": "What does 'triple-negative' breast cancer mean?", "input": "", "output": "Triple-negative means the cancer cells don't have the hormone receptors (estrogen and progesterone) or the HER2 protein that many other breast cancers have. It's a form that tends to have a higher risk of recurrence than most other types, but it's also an active area of research with treatments like immunotherapy specifically developed for it. Your oncologist can walk you through what this diagnosis means for your specific treatment path."},
  {"instruction": "What is stage 4 or metastatic breast cancer?", "input": "", "output": "Metastatic (stage 4) breast cancer means the cancer has spread from the breast to other parts of the body, like the bones, liver, lungs, or brain, usually traveling through the blood or lymph system. This is understandably frightening news to receive, but treatments today focus on slowing the cancer's spread and helping you feel as well as possible for as long as possible. Please talk with your oncology team about what this means specifically for you \u2014 everyone's path looks different."},
  {"instruction": "My biopsy says DCIS. Do I have cancer?", "input": "", "output": "Ductal carcinoma in situ (DCIS) is sometimes called 'stage 0' or noninvasive breast cancer \u2014 it means abnormal cells are in the milk duct lining but haven't spread beyond that, so it isn't invasive cancer. It's still something your doctor will want to treat because it can develop into invasive cancer if left alone. This is a good question to bring directly to your doctor, since treatment for DCIS is usually less extensive than for invasive breast cancer."},
  {"instruction": "Is LCIS the same as breast cancer?", "input": "", "output": "No \u2014 lobular carcinoma in situ (LCIS) is not actually breast cancer itself, but it does mean you have a higher future risk of developing breast cancer. It's understandable to find this confusing since the name includes 'carcinoma.' Your doctor can explain what kind of monitoring or risk-reduction options make sense for you."},
  {"instruction": "What is Paget disease of the breast?", "input": "", "output": "Paget disease of the breast is a rare type of cancer that affects the skin of the nipple and the areola around it. It's different from more common breast cancers because of where it starts, but your care team will still work with you on a treatment plan suited to it. If you've noticed nipple changes, it's important to get them checked by a doctor."},
  {"instruction": "I found a lump but my doctor said it's a phyllodes tumor, not cancer. Should I still be worried?", "input": "", "output": "That's a fair thing to want clarity on \u2014 the good news is most phyllodes tumors are not cancerous. They start in the connective tissue of the breast and are usually treated with surgical removal even when benign, just to be thorough. If you have any doubts or new symptoms, don't hesitate to bring them up with your doctor again."},
  {"instruction": "What are the molecular subtypes of breast cancer and why do they matter?", "input": "", "output": "Molecular subtypes \u2014 like luminal A, luminal B, HER2-positive, and triple-negative \u2014 are based on whether the cancer cells have hormone receptors, the HER2 protein, or other biomarkers. Knowing the subtype helps your doctor choose which treatments, like hormone therapy or targeted therapy, are likely to work best for your specific cancer. Ask your oncologist which subtype you have and how it's shaping your treatment recommendations."},
  {"instruction": "Can breast cancer happen in more than one place in the breast?", "input": "", "output": "Yes, breast cancer can form in glandular tissue (ducts and lobules), fibrous and fatty tissue, the nipple, or even the blood and lymph vessels of the breast, and it can occur in one or both breasts. Where it starts partly determines what type of breast cancer it's classified as. If you have questions about your specific diagnosis location, your doctor and pathology report are the best source of clarity."},
  {"instruction": "Is breast cancer always a lump you can feel?", "input": "", "output": "Not always \u2014 early breast cancer often causes no symptoms at all, which is exactly why regular screening mammograms matter so much. When symptoms do appear, they can include a lump, but also changes in breast size or shape, nipple discharge, or skin changes like dimpling or redness. If you notice any unusual change, even without a lump, it's worth having your doctor take a look."},
  {"instruction": "How common is breast cancer really?", "input": "", "output": "Breast cancer is the second most common cancer in women after skin cancer, and on average, women have about a 1 in 8 chance of developing it in their lifetime. That statistic can feel alarming, but it also reflects how much research and screening infrastructure exists around this specific cancer. If you're thinking about your own personal risk, a doctor can walk you through a more individualized risk assessment."},
  {"instruction": "Can men get breast cancer?", "input": "", "output": "Yes, though it's rare \u2014 fewer than 1 in 100 breast cancers in the U.S. occur in men, and it tends to happen at a later age in men than in women. Most male breast cancers are ductal cancers, since male breasts have far fewer lobules than female breasts. If a man in your life has noticed a breast lump or change, it's worth having a doctor evaluate it rather than assuming it can't be cancer."},
  {"instruction": "What is HER2-positive breast cancer?", "input": "", "output": "HER2-positive means the cancer cells have too much of a protein called HER2, which causes them to grow and divide faster than normal. This subtype has several targeted therapy drugs specifically developed for it, like trastuzumab, which can be very effective. Your oncologist can explain how a HER2-positive diagnosis shapes your particular treatment plan."},
  {"instruction": "What does hormone receptor-positive breast cancer mean?", "input": "", "output": "Hormone receptor-positive (HR-positive) means the cancer cells have receptors for estrogen and/or progesterone, which the cancer can use to grow. About 80% of people diagnosed with breast cancer have HR-positive cancer, and it typically responds well to hormone therapy. Ask your doctor whether hormone therapy is part of your recommended treatment plan."},
  {"instruction": "Are all breast lumps cancer?", "input": "", "output": "No \u2014 most breast changes, including lumps, are not cancer. That said, it's still important to have any new lump or unusual change checked by a doctor, since only proper testing can tell the difference between a benign condition and something that needs treatment. Please don't hesitate to get it looked at, even if you're fairly sure it's nothing serious."},
  {"instruction": "What's the difference between invasive and non-invasive breast cancer?", "input": "", "output": "Non-invasive (in situ) cancer means the abnormal cells are still contained within the ducts or lobules and haven't spread into surrounding breast tissue. Invasive cancer has spread beyond where it started and can potentially reach nearby lymph nodes or other organs. Most breast cancers diagnosed are invasive, but your pathology report will specify exactly which type you have."},
  {"instruction": "What is angiosarcoma of the breast?", "input": "", "output": "Angiosarcoma is a rare type of breast cancer that starts in the cells lining the blood vessels or lymph vessels of the breast. Because it's uncommon, your care team may involve specialists experienced with this specific type to guide your treatment. It's completely reasonable to ask your doctor for a clear explanation of what this diagnosis means for you specifically, since it's less commonly discussed than other breast cancer types."},

  {"instruction": "What actually causes breast cancer?", "input": "", "output": "Breast cancer is caused by changes in how breast cells grow and divide, but there's rarely one single cause \u2014 it's usually a combination of risk factors, some you can change (like alcohol use) and some you can't (like age, genetics, and family history). Having risk factors doesn't guarantee you'll develop breast cancer, and some people without any known risk factors still do. If you're worried about your personal risk, a doctor can help you understand it more precisely."},
  {"instruction": "Does having a family history of breast cancer mean I'll get it too?", "input": "", "output": "Not necessarily \u2014 having a family history increases your risk, but it doesn't mean it's certain you'll develop breast cancer. Genetic counselors can help you understand your personal likelihood based on your family history and discuss whether genetic testing makes sense for you. This is exactly the kind of conversation worth having with your doctor so you can make informed decisions about monitoring or prevention."},
  {"instruction": "What is BRCA1 and BRCA2 and should I get tested?", "input": "", "output": "BRCA1 and BRCA2 are genes that, when they carry certain harmful changes, significantly raise the risk of breast cancer (and some other cancers). Genetic testing might be recommended if you have a family history of these gene changes, were diagnosed with breast cancer before age 50, or have certain other risk factors. Whether testing is right for you is a personal decision best made with a genetic counselor or your doctor, who can walk you through the risks and benefits of knowing your results."},
  {"instruction": "I'm scared because my mom and my aunt both had breast cancer. What should I do?", "input": "", "output": "That's a completely understandable worry, and family history is one of the recognized risk factors for breast cancer. It would be worth talking to a genetic counselor or your doctor about your family history in detail \u2014 they can help assess your personal risk and discuss options like earlier or more frequent screening, or genetic testing. You're taking the right first step by paying attention to this."},
  {"instruction": "Does drinking alcohol really increase breast cancer risk?", "input": "", "output": "Yes, alcohol consumption is a recognized risk factor for breast cancer, and the risk increases the more alcohol you drink. It's one of the risk factors that's actually within your control to change. If you have concerns about your personal risk level, your doctor can help put this in context alongside your other risk factors."},
  {"instruction": "Can birth control pills cause breast cancer?", "input": "", "output": "The evidence here isn't fully settled \u2014 some studies suggest a slight increase in breast cancer risk for people currently or recently using hormonal birth control pills, while other studies haven't found an increased risk. It's a nuanced, individual decision that's worth discussing directly with your doctor, who can weigh this alongside your other risk factors and the benefits birth control provides you."},
  {"instruction": "I have dense breasts. Does that mean I'm more likely to get breast cancer?", "input": "", "output": "Having dense breast tissue is linked to a higher risk of breast cancer, and it can also make it harder to detect cancer on a standard mammogram. Because of this, your doctor might discuss additional screening options like breast MRI or ultrasound alongside your mammogram. It's worth asking your doctor directly what your breast density means for your personal screening plan."},
  {"instruction": "Does being overweight increase breast cancer risk?", "input": "", "output": "Yes, having excess body weight, especially after menopause, is a recognized risk factor for breast cancer. This is one of the modifiable risk factors, meaning lifestyle changes can potentially help lower risk over time. If you'd like guidance on this, it's best discussed with your doctor, who can help you set realistic and healthy goals rather than focusing on weight alone."},
  {"instruction": "I never breastfed my kids. Does that raise my breast cancer risk?", "input": "", "output": "Reproductive history does play a role in risk \u2014 factors like never having breastfed, never carrying a pregnancy to term, or having a longer lifetime exposure to estrogen (from early periods or late menopause, for example) are linked to somewhat higher risk. This doesn't mean you did anything wrong or that you're destined to develop breast cancer \u2014 it's just one piece of a bigger risk picture. If it's weighing on you, talking it through with your doctor can offer more personalized reassurance."},
  {"instruction": "Is breast cancer risk different by race?", "input": "", "output": "Yes, breast cancer incidence and outcomes do vary by race \u2014 in the U.S., White women have the highest rate of being diagnosed with breast cancer, while Black women have the highest death rate from it. This disparity reflects a complex mix of factors including access to care, and it's an active area of research and public health focus. If you have questions about how this applies to your own care, your doctor is a good place to start."},
  {"instruction": "What does it mean if my risk calculator says I have a 1 in 8 chance of breast cancer?", "input": "", "output": "That figure, roughly 1 in 8 or about 13%, actually reflects the average lifetime risk for women in general, not necessarily your personal risk specifically. Risk calculators use factors like your age at first period, age at first childbirth, and family history to give a more individualized estimate, though even these are best guesses based on current knowledge, not guarantees. Your doctor can walk you through what your specific number means and what, if anything, it suggests for your screening plan."},
  {"instruction": "Can radiation exposure as a child cause breast cancer later?", "input": "", "output": "Yes, having been exposed to high levels of radiation, particularly at a young age such as during treatment for childhood cancer, is a recognized risk factor for breast cancer later in life. If this applies to you, it's worth mentioning to your doctor so they can factor it into your screening recommendations, which may start earlier or be more frequent than average."},
  {"instruction": "If I don't have any risk factors, can I still get breast cancer?", "input": "", "output": "Unfortunately, yes \u2014 some people who develop breast cancer have no known risk factors at all, and having risk factors doesn't guarantee you'll develop it either. This is exactly why regular screening matters for everyone, not just those considered high-risk. If you're due for a mammogram, it's worth keeping up with routine screening regardless of your risk factor profile."},
  {"instruction": "What's the single biggest risk factor for breast cancer?", "input": "", "output": "Aside from being female, the most significant risk factor for breast cancer is simply getting older \u2014 risk increases with age. Other major factors include certain genetic mutations (like BRCA1/BRCA2), a strong family history, and prior chest radiation, which on their own can raise lifetime risk to 60% or more. If you're concerned about where you fall on this spectrum, a doctor can help you get a clearer, personalized picture."},
  {"instruction": "Can hormone replacement therapy for menopause cause breast cancer?", "input": "", "output": "Taking combination (estrogen plus progestin) hormone replacement therapy for menopause symptoms is linked to an increased risk of breast cancer. This doesn't mean HRT is off the table for everyone, but it's an important conversation to have with your doctor, who can help weigh the benefits for your menopause symptoms against this risk."},
  {"instruction": "Is it my fault if I get breast cancer because of my lifestyle?", "input": "", "output": "No, please don't think of it that way. While some risk factors like alcohol use or physical activity are within your control, most breast cancer risk comes from things you can't change \u2014 your age, genetics, and family history \u2014 and many people who develop breast cancer have few or no controllable risk factors at all. If you're processing a diagnosis, it can help to talk with your care team or a counselor about these feelings rather than carrying blame alone."},

  {"instruction": "What are the warning signs of breast cancer I should watch for?", "input": "", "output": "Key things to watch for include a new lump in the breast or underarm, changes in breast size or shape, nipple discharge that isn't breast milk, changes in the nipple's shape or direction, and skin changes like redness, dimpling, or scaliness. Most breast changes turn out not to be cancer, but any of these are worth having a doctor check out. Please don't wait if something feels off \u2014 it's always better to get reassurance than to wonder."},
  {"instruction": "Is breast pain a sign of cancer?", "input": "", "output": "Usually not \u2014 breast cancer doesn't typically cause pain, and breast soreness is more commonly caused by things like cysts, hormonal changes around your period, or certain medications. That said, if breast pain persists, it's still worth seeing a doctor to rule out other causes and get proper reassurance."},
  {"instruction": "I found a lump under my arm, not in my breast. Could that still be breast cancer?", "input": "", "output": "Yes, a lump under the arm is actually one of the recognized possible signs of breast cancer, since it can indicate spread to nearby lymph nodes. It's important to get this checked by a doctor rather than assuming it's unrelated to breast health. Try not to jump to conclusions before you've had it properly evaluated \u2014 many underarm lumps have benign causes too."},
  {"instruction": "My nipple has started pointing a different direction than before. Should I worry?", "input": "", "output": "A change in the direction your nipple points, or flattening of the nipple, is listed among the possible signs of breast cancer, so it's worth having a doctor take a look. Most breast changes are not cancer, but this is exactly the kind of change that deserves a proper evaluation rather than a wait-and-see approach. Please reach out to your doctor soon to get this checked."},
  {"instruction": "I have dimpling on my breast skin, like an orange peel texture. What could this mean?", "input": "", "output": "Dimpling or puckering of the breast skin is one of the listed skin changes that can be a sign of breast cancer, so this is worth having evaluated by a doctor promptly. It's understandable to feel anxious noticing this, but only a proper exam and possibly imaging can tell you what's actually going on. Please don't delay reaching out to your doctor."},
  {"instruction": "What are symptoms of breast cancer that has spread to other parts of the body?", "input": "", "output": "Metastatic breast cancer symptoms depend on where it has spread, but can include back or bone pain, bone fractures, shortness of breath, a persistent dry cough, abdominal pain or swelling, jaundice, severe headaches, seizures, or vision changes. If you're experiencing any of these, especially with a prior breast cancer history, please contact your doctor promptly \u2014 they can help determine the cause and next steps."},
  {"instruction": "I have redness and swelling on my breast but no lump. Could it still be cancer?", "input": "", "output": "Yes, general swelling or redness on the breast can be a sign of breast cancer even without a lump present \u2014 this is actually characteristic of inflammatory breast cancer, a less common but more aggressive form. It's important to have this checked promptly by a doctor, since inflammatory breast cancer can sometimes be mistaken for an infection at first. Please don't wait to get it evaluated."},
  {"instruction": "What should I do if I notice a breast change?", "input": "", "output": "Follow up with your doctor, even if you recently had a normal mammogram \u2014 breast changes are common and most aren't cancer, but only a proper evaluation can tell you for sure. It's always better to check something out and get reassurance than to leave it unexamined. Your doctor would rather see you and rule things out than have you wait."},
  {"instruction": "Can early breast cancer have no symptoms at all?", "input": "", "output": "Yes, early breast cancer very often causes no symptoms whatsoever, which is exactly why regular screening mammograms are so important \u2014 they can catch cancer before it becomes noticeable through symptoms. Please keep up with your recommended screening schedule even if you're feeling completely fine."},
  {"instruction": "My nipple has been itchy and slightly scaly for weeks. Is this something to worry about?", "input": "", "output": "Itching, tingling, or scaly skin on the nipple or areola are listed among the possible skin-related signs of breast cancer, including a rare type called Paget disease of the breast. It's understandably worrying, but this could also have simpler, benign explanations like dermatitis. Please have a doctor examine it to determine the cause and get you peace of mind either way."},

  {"instruction": "How does breast cancer screening actually help?", "input": "", "output": "Screening looks for breast cancer in people who don't yet have symptoms, which allows it to be found at an earlier, often more treatable stage. Regular screening mammograms have been shown to reduce deaths from breast cancer, making them an important part of routine care for most women. If you're unsure when you should start or how often to be screened, your doctor can guide you based on your personal risk factors."},
  {"instruction": "Is a mammogram the only way to screen for breast cancer?", "input": "", "output": "Mammography is the standard screening test for most women, but breast MRI and ultrasound may be added for people at higher risk or with dense breasts. Clinical breast exams and self-exams can help you stay aware of changes, but they aren't considered adequate as standalone screening tests. Talk with your doctor about which combination of tests makes sense for your personal risk level."},
  {"instruction": "Are there downsides to getting screened for breast cancer?", "input": "", "output": "Yes, screening does have potential harms alongside its benefits \u2014 these include false-positive results that can cause emotional distress and lead to unnecessary follow-up procedures, overdiagnosis of cancers that may never have caused problems, and a small amount of radiation exposure from mammography. It's a reasonable thing to discuss with your doctor so you understand both the benefits and limitations of screening for your situation."},
  {"instruction": "What's the difference between a screening mammogram and a diagnostic mammogram?", "input": "", "output": "A screening mammogram checks for cancer in people without symptoms, while a diagnostic mammogram is used after a lump or other symptom has been found, or to follow up on something spotted during a screening mammogram. Diagnostic mammograms usually involve more detailed pictures from different angles to closely examine the area of concern. If you've been asked to come back for a diagnostic mammogram, try not to panic \u2014 it's a standard next step, not a diagnosis in itself."},
  {"instruction": "Does breast MRI hurt or use radiation?", "input": "", "output": "Breast MRI doesn't use radiation \u2014 it uses radio waves and a strong magnet to create detailed images. It's more sensitive than mammography at detecting cancer, which is why it's sometimes recommended alongside mammography for people at high risk. Your doctor can tell you whether MRI screening makes sense as part of your personal screening plan."},
  {"instruction": "I have dense breasts \u2014 should I get an ultrasound too?", "input": "", "output": "If you have dense breasts, you may be offered a breast ultrasound in addition to mammography, since dense tissue can make mammograms harder to interpret. That said, it isn't yet clear whether this additional screening actually leads to better health outcomes, so this is a good conversation to have directly with your doctor about what's right for you."},
  {"instruction": "Is a breast self-exam enough to catch cancer early?", "input": "", "output": "Breast self-exams alone aren't considered an adequate screening test, but staying familiar with how your breasts normally look and feel is still valuable, since it helps you notice unusual changes to report to your doctor. Self-exams work best as a complement to regular mammograms, not a replacement for them."},
  {"instruction": "What happens if my mammogram comes back abnormal?", "input": "", "output": "An abnormal mammogram doesn't automatically mean cancer \u2014 false positives are actually fairly common, especially in younger women or those with dense breasts. Your doctor will likely recommend follow-up tests, like a diagnostic mammogram, ultrasound, or biopsy, to figure out what's actually going on. Try to take it one step at a time, and lean on your care team to walk you through what each next step means."},
  {"instruction": "How often should I get a mammogram?", "input": "", "output": "Recommended mammogram frequency depends on your age, personal risk factors, and current guidelines, which can vary somewhat between organizations. This is really best discussed directly with your doctor, since they can factor in your specific risk profile and history to recommend the right schedule for you."},
  {"instruction": "Can a clinical breast exam replace a mammogram?", "input": "", "output": "No, a clinical breast exam by itself is not considered an adequate screening test for breast cancer on its own. It can still be a useful part of your overall care, but it works best alongside, not instead of, regular mammography. Your doctor can explain how these different tools fit together in your screening plan."},

  {"instruction": "How is breast cancer actually diagnosed?", "input": "", "output": "Diagnosis usually starts with your doctor reviewing your symptoms and medical history, followed by a physical exam and imaging tests like a diagnostic mammogram, ultrasound, or MRI. If imaging finds something concerning, a biopsy is the only sure way to confirm whether it's cancer. Try to take the process one step at a time \u2014 your doctor will guide you through each test and what it means."},
  {"instruction": "What is a biopsy and will it hurt?", "input": "", "output": "A biopsy is a procedure where a surgeon removes a small sample of tissue or cells so a pathologist can examine them under a microscope \u2014 it's the only definitive way to diagnose breast cancer. There are several types, from needle biopsies that don't usually require anesthesia to surgical biopsies that do; most are done as outpatient procedures. Your doctor can walk you through exactly what to expect for your specific type of biopsy, including pain management."},
  {"instruction": "What's the difference between fine-needle and core-needle biopsy?", "input": "", "output": "Fine-needle aspiration uses a thin needle to remove tissue or fluid, while core-needle biopsy uses a wider needle to remove larger tissue samples, sometimes called cores. Your doctor will choose the type based on factors like where the abnormal area is and what information they need. Both are typically done as outpatient procedures without needing much recovery time."},
  {"instruction": "How long does it take to get biopsy results back?", "input": "", "output": "Timing can vary depending on your provider and lab, so it's a good idea to ask your doctor directly when your results will be ready and how you'll be notified. Waiting for results is genuinely one of the hardest parts of this process emotionally, and it's completely normal to feel anxious. If you don't hear back within the expected timeframe, it's okay to follow up rather than wait in silence."},
  {"instruction": "My test results showed up in my patient portal before I talked to my doctor. Should I look at them?", "input": "", "output": "It can be really hard to resist looking, but it might be worth waiting to review results with your doctor when possible, since they can help you understand what the results actually mean and avoid confusion or unnecessary worry from reading them alone. If you do see them early and feel distressed, it's okay to reach out to your care team sooner rather than waiting for the scheduled appointment."},
  {"instruction": "What is staging and why does it matter?", "input": "", "output": "Staging is the process of determining how far cancer has spread, based on factors like tumor size, whether it's reached lymph nodes, and biomarker test results. Knowing the stage helps your doctor recommend the treatment plan most likely to work well for your specific situation. Your doctor can explain your particular stage and what it means in plain terms whenever you're ready to talk through it."},
  {"instruction": "What is a sentinel lymph node biopsy?", "input": "", "output": "A sentinel lymph node biopsy checks the first lymph node(s) that cancer cells are most likely to spread to from the tumor, helping doctors determine if the cancer has spread beyond the breast. A special dye or radioactive substance is used to locate this node, which is then removed and examined by a pathologist. It's typically done as an outpatient procedure, often at the same time as breast surgery."},
  {"instruction": "What's a pathology report and what should I look for in it?", "input": "", "output": "A pathology report details what the pathologist found when examining your biopsy sample, including where the cancer started, its tumor grade, and whether it's invasive. It can be dense and full of medical terminology, so it's completely reasonable to ask your doctor to walk through it with you line by line rather than trying to interpret it alone."},
  {"instruction": "Should I get a second opinion on my breast cancer diagnosis?", "input": "", "output": "It's a completely reasonable thing to want, and many people do seek a second opinion to confirm a diagnosis or treatment plan. You'll need to share your test results and reports from your first doctor with the second one for them to review. Getting a second opinion doesn't mean you don't trust your doctor \u2014 it's a normal part of making sure you feel confident in your care plan."},
  {"instruction": "What does tumor grade mean?", "input": "", "output": "Tumor grade describes how abnormal the cancer cells look under a microscope and how quickly they're likely to grow and spread, scored from 1 (closer to normal, slower growing) to 3 (more abnormal, faster growing). A higher grade generally means the cancer is more likely to grow and spread quickly, which can influence treatment decisions like whether chemotherapy is recommended. Ask your doctor what your specific grade means for your treatment plan."},
  {"instruction": "What are multigene tests like Oncotype DX and why would I need one?", "input": "", "output": "Multigene tests, like Oncotype DX and MammaPrint, look at the activity of specific genes in your breast cancer cells to help predict the chance the cancer will spread or come back, and can help guide decisions about whether chemotherapy would actually be beneficial for you. If these tests show a low risk of recurrence, some people can safely skip chemotherapy. Your doctor can explain whether one of these tests is relevant to your treatment planning."},
  {"instruction": "What biomarker tests will be done on my breast cancer?", "input": "", "output": "All breast cancers are tested for biomarkers including estrogen receptors, progesterone receptors, and HER2 status, since these results help determine both the stage of the cancer and the best treatment approach. This testing is a standard part of diagnosis, not something extra to worry about. Your doctor will explain your specific biomarker results and how they shape your treatment recommendations."},

  {"instruction": "What treatment will I need for breast cancer?", "input": "", "output": "Treatment depends on many factors including the type and stage of your cancer, your biomarker results, your age and health, and your personal preferences \u2014 most people receive a combination of treatments like surgery, radiation, chemotherapy, hormone therapy, or targeted therapy. There's no single standard plan, which is exactly why your care team will build one specifically around your situation. It's okay to ask lots of questions as they walk you through your options."},
  {"instruction": "What's the difference between a lumpectomy and a mastectomy?", "input": "", "output": "A lumpectomy, also called breast-conserving surgery, removes just the tumor and some surrounding tissue, while a mastectomy removes the whole breast. Lumpectomy is often used when possible, but mastectomy may be recommended if the tumor is large or found in multiple areas. Your surgeon can help you understand which option, along with reconstruction choices if relevant, fits your particular case best."},
  {"instruction": "Will I need radiation after surgery for breast cancer?", "input": "", "output": "Often, yes \u2014 radiation therapy is commonly given after a lumpectomy to reduce the chance of cancer returning in the breast, though it may not be needed after a mastectomy. The exact recommendation depends on your specific case, including tumor size and whether cancer reached the lymph nodes. Your radiation oncologist can explain exactly what to expect if it's part of your plan."},
  {"instruction": "What are the side effects of radiation therapy for breast cancer?", "input": "", "output": "Radiation side effects can include skin changes like redness or dryness in the treated area, breast tenderness or hardening, and fatigue during treatment. Some effects, called late effects, can appear months or years later and may include heart, lung, or bone issues, or rarely, a second cancer. Talk with your doctor or nurse about what to expect and how supportive care can help you manage side effects as they come up."},
  {"instruction": "Will I definitely need chemotherapy for breast cancer?", "input": "", "output": "Not necessarily \u2014 not everyone with breast cancer receives chemotherapy, and multigene tests like Oncotype DX can sometimes show that you're unlikely to benefit from it, meaning you can skip it without increasing your recurrence risk. Whether chemo is recommended depends on factors like tumor grade, lymph node involvement, and hormone receptor or HER2 status. Your oncologist can explain the reasoning behind your specific recommendation."},
  {"instruction": "What are the most common side effects of chemotherapy?", "input": "", "output": "The most common side effect is fatigue, along with others like hair loss, mouth sores, and nausea. Not everyone experiences the same side effects even with the same treatment, and your care team has ways to help prevent or manage many of them. Please tell your care team about any side effects you're having \u2014 there's often something that can help."},
  {"instruction": "What is hormone therapy and who needs it?", "input": "", "output": "Hormone therapy slows or stops the growth of breast cancers that have hormone receptors (estrogen and/or progesterone receptors) by blocking the body's hormone production or its effects on cancer cells. About 80% of people diagnosed with breast cancer have hormone receptor-positive cancer and may benefit from this treatment. Your doctor will let you know if your specific biomarker results make you a candidate for hormone therapy."},
  {"instruction": "What is targeted therapy and how is it different from chemotherapy?", "input": "", "output": "Unlike chemotherapy, which affects rapidly dividing cells throughout the body, targeted therapy specifically targets proteins that control how cancer cells grow, divide, and spread \u2014 for example, drugs like trastuzumab specifically target HER2-positive cancers. It may be used for HER2-positive, hormone receptor-positive, triple-negative, or BRCA-related breast cancers. Your oncologist can explain whether your biomarker results make you eligible for a targeted therapy option."},
  {"instruction": "What is immunotherapy and will I need it?", "input": "", "output": "Immunotherapy helps your own immune system fight cancer more effectively, and for breast cancer it's primarily used to treat triple-negative breast cancer, often alongside chemotherapy. It's a newer treatment approach compared to chemotherapy or surgery, and not everyone with breast cancer will be a candidate. Your care team can tell you whether it's part of the recommended plan for your specific diagnosis."},
  {"instruction": "Why would I get chemo before surgery instead of after?", "input": "", "output": "Chemotherapy before surgery, called neoadjuvant chemotherapy, is sometimes used to shrink a large tumor so that breast-conserving surgery becomes possible instead of a mastectomy, or to reduce the number of lymph nodes that need removal. Your doctor may recommend this approach for high-grade, HER2-positive, or triple-negative cancers, among other reasons. It's worth asking your oncologist why this sequence is being recommended in your specific case."},
  {"instruction": "What is palliative care and does it mean giving up?", "input": "", "output": "No, palliative care does not mean giving up \u2014 it's care focused on improving quality of life by managing symptoms and providing support, and it can be given at any stage of disease alongside treatments aimed at fighting the cancer, like chemotherapy. It's about helping you feel as well as possible throughout treatment, not a sign that treatment has stopped working. If it's offered to you, it may be worth asking your care team more about what it could offer you specifically."},
  {"instruction": "Can breast cancer come back after treatment?", "input": "", "output": "Yes, breast cancer can sometimes return after treatment, either locally in the breast, chest wall, or nearby lymph nodes, or in other parts of the body. This is exactly why follow-up care and regular check-ups matter so much after treatment ends. It's very normal to feel anxious about recurrence \u2014 many survivors do, and it's worth talking to your care team about how to manage that fear as well as the physical monitoring."},
  {"instruction": "How is metastatic breast cancer treated differently?", "input": "", "output": "Treatment for metastatic (stage 4) breast cancer focuses on slowing the cancer's spread and controlling symptoms rather than eliminating it entirely, and may use some of the same treatments as earlier stages depending on how the cancer has already responded to prior treatment. Choices are also guided by what matters most to you in terms of quality of life and goals of care. This is a deeply personal conversation to have with your oncology team, who can help align treatment with what you want."},

  {"instruction": "How do I cope with the fear and stress of a breast cancer diagnosis?", "input": "", "output": "It's completely normal for a breast cancer diagnosis to bring up intense emotions, and you don't have to manage them alone \u2014 your care team, including social workers, can offer real support and resources. Many people find it helpful to lean on family, friends, or support groups specifically for people going through cancer treatment. Please don't hesitate to ask your care team for these kinds of resources; support is available.\n\nIf you're finding this especially overwhelming, it can also help to talk with a mental health professional who specializes in supporting people through serious illness."},
  {"instruction": "Will I need follow-up appointments after breast cancer treatment ends?", "input": "", "output": "Yes, most breast cancer survivors need regular follow-up exams and tests, which your doctor will typically outline in a survivorship care plan. These visits may include physical exams, mammograms, or other imaging to watch for any signs of recurrence. It's normal for scheduling these to feel stressful or like a lot to manage, but they're an important part of staying supported after treatment."},
  {"instruction": "Why am I so anxious about my scans even though treatment worked?", "input": "", "output": "This is a very common experience, sometimes called 'scanxiety' \u2014 it's completely normal to feel a mix of relief and fear once active treatment ends, especially around follow-up tests and scans. These feelings often do improve with time and support, but they can also feel very real and consuming in the moment. It's worth mentioning this anxiety to your care team, since they may have specific resources or strategies to help you cope."},
  {"instruction": "How will breast cancer treatment affect my body image?", "input": "", "output": "Breast surgery, reconstruction decisions, and side effects like hair loss can genuinely affect how you see and feel about your body, and those feelings are valid, whatever they are. There are resources specifically designed to help with these changes, and it can help to talk to your care team or a counselor about what you're experiencing. You're allowed to grieve these changes even while focusing on getting better."},
  {"instruction": "What late effects should I watch for after finishing treatment?", "input": "", "output": "Late effects are problems that can show up months or years after treatment ends, and they vary depending on which treatments you received \u2014 they might include things like heart or lung issues from radiation, or lymphedema from lymph node removal. Your survivorship care plan should outline what to watch for based on your specific treatments. If you notice new symptoms after treatment, it's worth mentioning them to your doctor rather than assuming they're unrelated."},
  {"instruction": "How can I support my partner or family member going through breast cancer treatment?", "input": "", "output": "Caregiving is genuinely demanding, and it's important to know that support resources exist for caregivers too, not just patients. Learning about what to expect during treatment can help you feel more prepared, and staying connected with your loved one's care team can help you understand how best to support them. Please also take care of your own needs \u2014 caregiver burnout is real, and you deserve support as well."},
  {"instruction": "I'm worried about the cost of breast cancer treatment. What can I do?", "input": "", "output": "Financial stress during cancer treatment is very common, even for people with health insurance, so please know you're not alone in worrying about this. There are resources and support programs specifically designed to help manage the costs of treatment, and it's worth asking your care team, particularly a social worker, about what's available. Bringing this concern up with your care team early can help you access support sooner rather than facing it alone."},
  {"instruction": "Is it normal to feel like my emotions are all over the place during treatment?", "input": "", "output": "Yes, this is very normal \u2014 cancer and its treatment can bring up a wide range of emotions you might not be used to dealing with, and existing feelings can feel more intense too. You're not overreacting, and these feelings deserve support just as much as the physical side effects do. Your care team can connect you with resources to help you process what you're feeling, and it's okay to ask for that support."},
  {"instruction": "What questions should I ask my doctor at my next appointment?", "input": "", "output": "It can help to write questions down beforehand so you don't forget them in the moment \u2014 things like what your specific diagnosis and stage mean, what treatment options are recommended and why, what side effects to expect, and what your follow-up care will look like are all reasonable to ask. There are also resource guides specifically about questions to ask your doctor about cancer that you might find helpful to review beforehand. Don't feel like any question is too small \u2014 your care team wants you to understand your own care."},

  {"instruction": "Is breast cancer during pregnancy treated differently?", "input": "", "output": "Yes, treatment during pregnancy requires careful adjustments \u2014 for example, chemotherapy is generally avoided during the first three months but may be given later in pregnancy with care taken around timing near delivery, and breastfeeding is not recommended during chemotherapy since the drugs can pass into breast milk. Surgery, like lumpectomy or mastectomy, is often still an option during pregnancy. This is a complex and emotional situation that really needs to be managed closely with a specialized care team who can balance your health and your baby's."},
  {"instruction": "My husband found a lump in his breast. Is that even possible?", "input": "", "output": "Yes, men can get breast cancer, even though it's much less common than in women \u2014 fewer than 1 in 100 breast cancers in the U.S. occur in men. It's important he gets it checked by a doctor rather than dismissing it, since male breast cancer is real and treatable when caught. Please encourage him to make an appointment rather than waiting."},
  {"instruction": "Is it safe to get treated for breast cancer while pregnant?", "input": "", "output": "It can be, though it requires very careful coordination between your oncology and obstetric teams to protect both you and the baby. Certain treatments and timings are adjusted specifically for pregnancy, and your care team will design a plan aimed at treating the cancer effectively while minimizing risk to your pregnancy. This is an area where you'll want a specialized team experienced in treating cancer during pregnancy, so please make sure that's part of your care."},
  {"instruction": "How is male breast cancer different from female breast cancer?", "input": "", "output": "Male breast cancer most often starts in the milk ducts and is called ductal cancer, since men have far fewer milk-producing lobules than women do, making lobular cancer rare in men. It tends to occur at a later age in men than in women. Diagnosis and treatment approaches share a lot in common with female breast cancer, but your doctor can explain any differences relevant to your specific case."},
  {"instruction": "Should I stop breastfeeding if I'm diagnosed with breast cancer?", "input": "", "output": "If surgery is planned, breastfeeding is often stopped beforehand to reduce blood flow and make the breasts smaller for the procedure, and breastfeeding is not recommended during chemotherapy since the drugs can pass into breast milk and potentially harm the baby. This is understandably a difficult adjustment, and your oncology team can help you navigate the timeline and any concerns about feeding your baby safely during treatment."},

  {"instruction": "What does stage 1 breast cancer mean in simple terms?", "input": "", "output": "Stage I generally means the cancer is small and hasn't spread far, if at all, beyond the breast \u2014 it's considered an early stage, which usually means more treatment options and a very favorable outlook. The exact staging also factors in things like tumor grade and hormone/HER2 status, which is why two people both called 'stage 1' can still have somewhat different treatment plans. Your doctor can explain exactly what your specific stage means for you."},
  {"instruction": "What's the difference between stage 2 and stage 3 breast cancer?", "input": "", "output": "In general, stage II usually means a somewhat larger tumor and/or limited spread to nearby lymph nodes, while stage III (often called locally advanced) usually means more extensive spread to lymph nodes or nearby tissue, though it still hasn't spread to distant parts of the body. Staging also takes into account tumor grade and biomarker status, not just size. Your care team can walk you through exactly what your stage means and how it's shaping your treatment plan."},
  {"instruction": "Is stage 4 breast cancer curable?", "input": "", "output": "Stage 4 (metastatic) breast cancer is generally not considered curable with current treatments, but it is treatable \u2014 many people live for years with ongoing treatment focused on controlling the cancer and managing symptoms. This is an incredibly hard thing to sit with, and it's completely okay to feel overwhelmed by it. Please talk openly with your oncology team about your prognosis and what matters most to you in your care, and consider leaning on support resources too."},
  {"instruction": "What does TNM staging mean?", "input": "", "output": "TNM staging is a system that looks at three main factors: T (the size and extent of the tumor), N (whether it has reached nearby lymph nodes), and M (whether it has spread, or metastasized, to distant parts of the body). These are combined with grade and biomarker results to determine your overall stage. It can look like a confusing string of letters and numbers on paper, so don't hesitate to ask your doctor to translate it into what it actually means for you."},
  {"instruction": "What happens if breast cancer comes back after treatment?", "input": "", "output": "If cancer returns, it's called a recurrence, and it can come back locally in the breast, regionally in nearby lymph nodes or skin, or at a distant site like the liver, lungs, or bone. Your doctor will likely repeat many of the same tests used at your original diagnosis to figure out where and how far it has returned, sometimes called 'restaging.' This news is understandably very hard to receive, and it's important to lean on your care team and support network as you talk through next steps."},
  {"instruction": "What are survival rates for breast cancer?", "input": "", "output": "Survival statistics vary a lot depending on the type and stage of breast cancer \u2014 for example, cancer found only in the breast (localized) has a much higher 5-year survival rate than cancer that has spread to distant parts of the body. It's important to remember that these numbers come from large groups of people and can't predict exactly what will happen to any one individual. Please talk with your doctor about how these statistics relate to your specific diagnosis, since they can add important context that a number alone can't."},
  {"instruction": "My doctor mentioned my '5-year survival rate.' What does that actually mean?", "input": "", "output": "The 5-year relative survival rate tells you what percentage of people with a similar type and stage of breast cancer are alive 5 years after diagnosis, compared to the general population \u2014 it's a statistical tool, not a personal prediction. These numbers also reflect treatments from several years ago, so they may not fully capture the benefit of newer treatments available today. It's very reasonable to ask your doctor to help you understand what this statistic does and doesn't tell you about your own situation."},
  {"instruction": "Can I trust breast cancer statistics to know what will happen to me?", "input": "", "output": "Not exactly \u2014 survival statistics are based on large groups of people over time, so they can't predict exactly what will happen for any one individual, and responses to treatment vary even among people with the same diagnosis. They can offer general context, but your specific prognosis depends on many personal factors that your doctor can help you understand. Try not to fixate too heavily on a single number \u2014 it's just one piece of a much bigger picture."},
  {"instruction": "What factors affect my personal breast cancer prognosis?", "input": "", "output": "Many things factor into prognosis, including the type of breast cancer, tumor size and spread, tumor grade, biomarker status (like hormone receptors and HER2), your age and overall health, and how your cancer responds to treatment. Because so many factors are involved, your prognosis is genuinely individual to you. Your doctor is the best person to help you understand how these pieces come together for your specific situation."},
  {"instruction": "What does it mean to be a 'breast cancer survivor'?", "input": "", "output": "The term 'survivor' is often used from the moment of diagnosis onward, through treatment and beyond, though what it means to you personally can change over time \u2014 and it's completely okay if the term doesn't resonate with you. There's no one right way to feel about this label or this journey. What matters most is finding language and community that feels right for you."},
  {"instruction": "Will I need breast reconstruction after a mastectomy?", "input": "", "output": "Reconstruction is an option, not a requirement \u2014 it's a personal decision, and some people choose it while others don't. If you're considering it, your surgical team can walk you through the different approaches and timing options, including whether it could be done at the same time as your mastectomy or later. This is a decision worth taking your time with and discussing thoroughly with your care team."},
  {"instruction": "How long does recovery take after a mastectomy versus a lumpectomy?", "input": "", "output": "Recovery time varies by procedure and by individual, but generally, mastectomy involves more extensive surgery and a longer recovery than lumpectomy, which is a less invasive, breast-conserving procedure. Whether you need general or local anesthesia and how extensive the surgery is will affect your specific recovery timeline. Your surgeon can give you a more precise recovery estimate based on exactly what procedure you're having."},
  {"instruction": "What are the harms of overdiagnosis in breast cancer screening?", "input": "", "output": "Overdiagnosis refers to finding cancers through screening that may never have caused symptoms or problems during a person's lifetime \u2014 finding these doesn't necessarily improve health outcomes but does expose the person to the risks and side effects of treatment they may not have needed. It's a genuinely complex trade-off in cancer screening that researchers continue to study. If this concerns you, it's worth discussing with your doctor how it factors into your personal screening decisions."},
  {"instruction": "Why do false-positive mammogram results happen?", "input": "", "output": "False positives happen more often in younger women, those with dense breasts, people with a history of previous breast biopsies, those with a family history of breast cancer, or people taking estrogen therapy. A false-positive doesn't mean anything went wrong with your mammogram \u2014 it just means something needed a closer look that turned out not to be cancer. It's still an emotionally difficult experience, and it's okay to feel shaken even after getting reassuring news."},
  {"instruction": "How is breast cancer treated if it can't be removed with surgery?", "input": "", "output": "When surgery isn't an option, treatment typically relies on systemic therapies like chemotherapy, hormone therapy, targeted therapy, or immunotherapy, sometimes in combination, to control the cancer and manage symptoms. The specific approach depends on your cancer's type, biomarkers, and how it has responded to prior treatments. Your oncology team can walk you through what this looks like for your specific situation."},
  {"instruction": "What's the difference between local, regional, and distant recurrence?", "input": "", "output": "Local recurrence means the cancer has come back in the breast itself; regional recurrence means it's returned in nearby lymph nodes, chest wall, or skin; and distant (or metastatic) recurrence means it has returned in another part of the body like the liver, lungs, or bones. Each type is evaluated and treated somewhat differently, so your doctor's restaging tests will help clarify exactly which type you're facing. However worrying this news is, your care team can help you understand your specific treatment options moving forward."},
  {"instruction": "Does a higher tumor grade always mean a worse prognosis?", "input": "", "output": "Higher tumor grade (meaning the cells look more abnormal and are likely to grow faster) is one factor that can affect prognosis, but it's just one piece of a much larger picture that includes tumor size, spread, biomarker status, and how you respond to treatment. It's not accurate to look at grade alone and draw conclusions about outcome. Your doctor can help you understand how your grade fits into your overall prognosis."},
  {"instruction": "Can I ask my doctor to explain my exact prognosis in plain language?", "input": "", "output": "Absolutely, and you should feel completely comfortable asking for that. Prognosis discussions can involve a lot of statistics and medical terminology, and it's entirely reasonable to ask your doctor to break it down in plain, clear language, or even to write key points down for you to review later. Understanding your prognosis, as much as you want to, can help you make more informed decisions about your treatment and your life."},

  {"instruction": "Where can I find reliable information about breast cancer online?", "input": "", "output": "The National Cancer Institute (cancer.gov) and the World Health Organization are strong starting points for accurate, evidence-based information on breast cancer. It's worth being cautious of sources that make dramatic claims or promote unproven treatments, and always bringing anything you read back to your own doctor to check how it applies to your situation. Your care team is ultimately your best resource for questions specific to your diagnosis."},
  {"instruction": "Is breast cancer contagious?", "input": "", "output": "No, breast cancer is not contagious \u2014 it can't be spread from person to person through contact, in the way an infection can. It develops from changes within a person's own cells over time. If you're worried about this for any reason, please know there's no risk to family, friends, or caregivers from being close to someone with breast cancer."},
  {"instruction": "Can stress cause breast cancer?", "input": "", "output": "There isn't strong scientific evidence that stress by itself directly causes breast cancer. That said, managing stress is still valuable for your overall wellbeing, especially while navigating a diagnosis or treatment. If you're carrying guilt about stress somehow causing your cancer, please know that's not something the evidence supports, and it may help to talk this through with your care team or a counselor."},
  {"instruction": "Does wearing an underwire bra cause breast cancer?", "input": "", "output": "No, there's no scientific evidence linking bra-wearing, including underwire bras, to breast cancer risk. This is a common myth, but it's not supported by research. Focus instead on the established risk factors and screening guidelines your doctor discusses with you."},
  {"instruction": "Can deodorant or antiperspirant cause breast cancer?", "input": "", "output": "No, there's no strong scientific evidence linking antiperspirant or deodorant use to breast cancer risk, despite this being a commonly circulated claim online. If you have specific product concerns, it's still fine to raise them with your doctor, but this particular link isn't supported by current research."},
  {"instruction": "If I have a mastectomy, will I definitely need chemotherapy too?", "input": "", "output": "Not necessarily \u2014 whether you need chemotherapy depends on factors like tumor grade, lymph node involvement, and biomarker or multigene test results, not simply on which surgery you have. Some people only need surgery, while others need a combination of treatments. Your oncologist can explain exactly why chemotherapy is or isn't recommended in your specific case."},
  {"instruction": "How soon after finding a lump should I see a doctor?", "input": "", "output": "As soon as you reasonably can \u2014 most breast lumps aren't cancer, but getting it checked promptly means you get answers and, if needed, can start treatment sooner rather than later. There's no benefit to waiting and worrying on your own. Please make an appointment rather than putting it off."},
  {"instruction": "Can young women get breast cancer, or is it just older women?", "input": "", "output": "While breast cancer mostly affects women aged 45 and older, younger women can and do get it too, though it's less common. If you're young and have noticed a breast change, it's still worth having it checked \u2014 age alone shouldn't be a reason to dismiss a symptom. Your doctor can evaluate any concerns regardless of your age."},
  {"instruction": "Is there a way to prevent breast cancer completely?", "input": "", "output": "Unfortunately, there's no guaranteed way to prevent breast cancer entirely, since many risk factors, like age and genetics, can't be changed. However, some modifiable factors, like limiting alcohol, staying physically active, and maintaining a healthy weight after menopause, may help lower risk somewhat. For people at very high risk due to genetics or family history, doctors may also discuss additional preventive options. Talk with your doctor about what risk-reduction strategies make sense for you personally."},
  {"instruction": "What's the difference between a benign breast lump and cancer?", "input": "", "output": "A benign lump is not cancerous and won't spread to other parts of the body, while a cancerous lump involves cells that grow uncontrollably and can potentially spread. The only way to know for certain which one you have is through proper evaluation, which may include imaging and sometimes a biopsy. Most breast lumps turn out to be benign, but it's always worth having any new lump checked by a doctor."},
  {"instruction": "My mammogram was normal but I still feel something unusual. Should I push for more testing?", "input": "", "output": "Yes, absolutely \u2014 it's important to let your doctor know about any unusual change you feel, even after a recent normal mammogram, since mammograms don't catch everything, particularly in dense breast tissue. Trust what you're feeling and advocate for further evaluation if something still feels off to you. You know your body, and it's always reasonable to ask for a closer look."},
  {"instruction": "What is a 'benign' breast condition and should I still be monitored?", "input": "", "output": "A benign condition means the change or lump found is not cancer, though some benign conditions, like LCIS or certain kinds of breast changes, can still be linked to a higher future risk of breast cancer and may warrant ongoing monitoring. Your doctor can tell you whether your specific benign finding needs any follow-up or additional screening going forward."},
  {"instruction": "Does having had DCIS in the past increase my future breast cancer risk?", "input": "", "output": "Yes, having had ductal carcinoma in situ (DCIS) in the past is a recognized risk factor for developing breast cancer again in the future. This is exactly why your doctor will likely recommend continued regular follow-up and screening even after DCIS treatment. If you have questions about your specific monitoring plan, it's worth bringing them up at your next appointment."},
  {"instruction": "Why does my doctor keep testing my hormone receptor and HER2 status even after diagnosis?", "input": "", "output": "These biomarker tests aren't just for the initial diagnosis \u2014 they help determine which treatments, like hormone therapy or targeted therapy, are most likely to work for your specific cancer, and they can also factor into your overall prognosis. Testing may be repeated at different points, such as if the cancer recurs, since biomarker status can sometimes change over time. If you're unsure why a particular test is being repeated, it's always fine to ask your doctor directly."},
  {"instruction": "What does it mean if my cancer is ER-positive and PR-negative?", "input": "", "output": "This means your cancer cells have estrogen receptors but not progesterone receptors, which still generally makes the cancer 'hormone receptor-positive' and a candidate for hormone therapy, though your doctor will factor in the specific combination when planning your treatment. These combinations, along with HER2 status, help define your cancer's molecular subtype. Ask your oncologist how your specific receptor status is shaping your treatment recommendations."},
  {"instruction": "Can breast cancer be found through a blood test?", "input": "", "output": "Not typically as a primary screening tool \u2014 mammography remains the standard way to screen for breast cancer, while blood tests are more commonly used for other purposes, like certain biomarker or genetic testing once a diagnosis is being worked up. Researchers continue to study blood-based tests for early cancer detection, but they aren't yet a replacement for imaging-based screening. Your doctor can explain what role, if any, blood tests play in your specific care."},
  {"instruction": "How does my doctor decide which chemotherapy drugs to use?", "input": "", "output": "The choice depends on several factors, including your cancer's grade, whether it's in the lymph nodes, and its HER2 or hormone receptor status, since different chemotherapy drugs and combinations work better for different cancer characteristics. Common drug classes include anthracyclines, taxanes, and platinum agents, sometimes combined with targeted therapy or immunotherapy depending on your subtype. Your oncologist can explain exactly why they've chosen your specific regimen."},
  {"instruction": "Why would I get radiation after a mastectomy if the whole breast was removed?", "input": "", "output": "While radiation is more commonly skipped after a mastectomy compared to a lumpectomy, it may still be recommended in certain situations, such as if the tumor was large, cancer was found in multiple lymph nodes, or other risk factors for recurrence are present. It's aimed at reducing the chance of cancer coming back in the chest wall area. Your radiation oncologist can explain the specific reasoning for recommending or not recommending it in your case."},
  {"instruction": "What's the earliest age breast cancer screening usually starts?", "input": "", "output": "Recommended starting ages for screening mammograms vary somewhat between different guideline organizations and depend on your personal risk factors, so this is best confirmed directly with your doctor. If you have higher-risk factors, like a strong family history or a known genetic mutation, your doctor may recommend starting screening earlier than average. It's worth having this specific conversation at your next check-up."},
  {"instruction": "Is chemotherapy the same for every type of breast cancer?", "input": "", "output": "No, chemotherapy drug choices and schedules vary depending on your cancer's grade, spread, and biomarker status \u2014 for example, HER2-positive cancer is often paired with targeted therapy alongside chemo, while triple-negative cancer might be combined with immunotherapy. There isn't a single one-size-fits-all chemotherapy protocol for breast cancer. Your oncologist will tailor your specific regimen to your diagnosis."},
  {"instruction": "What does 'adjuvant' and 'neoadjuvant' mean in cancer treatment?", "input": "", "output": "Adjuvant treatment is given after the main treatment, usually surgery, to help kill any remaining cancer cells and lower the risk of recurrence. Neoadjuvant treatment is given before surgery, often to shrink a tumor to make surgery more effective or less extensive. Your doctor can explain which approach, or combination, is being used in your specific treatment plan and why."},
  {"instruction": "What is tamoxifen and how does it work?", "input": "", "output": "Tamoxifen is a type of hormone therapy that blocks estrogen's effect on breast cancer cells, and it's one of the most established treatments for hormone receptor-positive breast cancer. Taken for 5 years or more, it can meaningfully reduce the risk of new breast cancer, recurrence, and death from breast cancer. Your doctor can tell you whether tamoxifen or another hormone therapy option is recommended for your specific diagnosis."},
  {"instruction": "What are aromatase inhibitors and who takes them?", "input": "", "output": "Aromatase inhibitors, like anastrozole, letrozole, and exemestane, are a type of hormone therapy that lowers estrogen levels in the body, and they're typically used in postmenopausal women with hormone receptor-positive breast cancer. They're sometimes combined with other targeted therapy drugs for added effectiveness. Your doctor can explain whether this is part of your recommended treatment plan based on your menopausal status and hormone receptor results."},
  {"instruction": "How long will I need to take hormone therapy for breast cancer?", "input": "", "output": "Hormone therapy is often recommended for 5 years or more, since research shows this duration can significantly reduce the risk of recurrence and improve outcomes for hormone receptor-positive breast cancer. The exact duration and specific drug depend on factors like your menopausal status and individual risk. Your doctor will tailor the length of treatment to your specific situation."},
  {"instruction": "What is ovarian ablation and why would I need it?", "input": "", "output": "Ovarian ablation is a way of blocking or suppressing ovarian function, which is the main source of estrogen in premenopausal women, in order to reduce or eliminate estrogen that could otherwise fuel hormone receptor-positive breast cancer. It's one of several hormone therapy approaches your doctor might consider depending on your menopausal status and specific treatment goals. This is worth discussing directly with your doctor, especially if you have questions about fertility or menopause-related effects."},
  {"instruction": "Will hormone therapy for breast cancer put me into menopause?", "input": "", "output": "Some hormone therapy approaches, particularly those that suppress ovarian function, can cause menopause-like symptoms or induce menopause in premenopausal women. This is an important and very personal topic to discuss with your doctor beforehand, especially if fertility preservation or family planning matters to you. Please raise these concerns early so your care team can help you weigh your options."},
  {"instruction": "What are common side effects of hormone therapy for breast cancer?", "input": "", "output": "Side effects can include hot flashes, joint pain, fatigue, and menopause-like symptoms, though they vary depending on the specific drug and individual. Some effects may ease over time, and your care team has strategies to help manage many of them. Please tell your doctor about any side effects you're experiencing, since there are often ways to make them more manageable."},
  {"instruction": "What is genetic counseling and what happens during a session?", "input": "", "output": "A genetic counselor reviews your personal and family medical history to assess your likelihood of carrying a gene change linked to breast cancer, and helps you understand your options for testing, including the potential risks and benefits of knowing your results. They can also support you in deciding how to share results with family members who might also be affected. It's a supportive, informative conversation, not just a lab test, and can be really valuable if you're weighing whether genetic testing is right for you."},
  {"instruction": "If I test positive for a BRCA mutation, does that mean I'll definitely get breast cancer?", "input": "", "output": "No, a positive BRCA test means you have a significantly higher risk of developing breast cancer, but it's not a certainty \u2014 some people with these mutations never develop cancer. Genetic counselors can help you understand your specific risk level and discuss options like increased screening or preventive measures. This is a lot to process, and it's completely okay to take time and lean on your care team as you figure out next steps."},
  {"instruction": "Should my siblings get tested if I test positive for a BRCA mutation?", "input": "", "output": "It's worth discussing with a genetic counselor, since first-degree relatives of someone with a known harmful gene change do have a higher chance of carrying that same change themselves. Genetic counselors can also help guide the sometimes difficult conversations about sharing this information with family members. This is a personal family decision, and there's no pressure to rush it \u2014 your genetic counselor can support you in navigating it at your own pace."},
  {"instruction": "What is external beam radiation therapy like day to day?", "input": "", "output": "External beam radiation therapy is usually given as an outpatient treatment, meaning you go to a clinic or radiation center for treatment and go home the same day, typically over a series of sessions rather than all at once. It targets radiation at the cancer or the area where the cancer was, aiming to spare healthy tissue as much as possible. Your radiation oncology team can walk you through exactly what your specific schedule and daily experience will look like."},
  {"instruction": "What is brachytherapy for breast cancer?", "input": "", "output": "Brachytherapy, or internal radiation therapy, involves placing radioactive material directly in the area where the tumor was removed, usually after breast-conserving surgery, so the radiation is concentrated where the recurrence risk is highest. It's an alternative to external beam radiation in certain cases. Your radiation oncologist can explain whether this approach fits your specific treatment plan."},
  {"instruction": "Will radiation therapy make me radioactive or unsafe to be around others?", "input": "", "output": "External beam radiation therapy does not make you radioactive \u2014 it's safe to be around others, including children, after your treatment sessions. Brachytherapy involves a temporary radioactive source placed in the treatment area, and your care team will give you specific instructions if any precautions are needed during that period. If you have concerns about safety around loved ones, it's completely reasonable to ask your radiation oncology team directly."},
  {"instruction": "Can breast cancer treatment affect my fertility?", "input": "", "output": "Yes, certain treatments like chemotherapy and some hormone therapies can affect fertility, which is understandably a difficult and important concern for many people facing a breast cancer diagnosis. If preserving fertility matters to you, it's important to raise this with your care team as early as possible, ideally before starting treatment, since there may be options like egg or embryo freezing to discuss. Please don't feel like this is too small a concern to bring up \u2014 it's a valid and significant part of your care planning."},
  {"instruction": "What is lymphedema and why does it happen after breast cancer treatment?", "input": "", "output": "Lymphedema is swelling, usually in the arm, that can occur when lymph nodes are removed or damaged during breast cancer surgery or radiation, disrupting normal lymph fluid drainage. It's a recognized possible side effect, and your care team can teach you ways to help reduce your risk and manage symptoms if it does develop. If you notice new or worsening swelling in your arm, please let your doctor know so they can help address it early."},
  {"instruction": "Is hair loss guaranteed with breast cancer chemotherapy?", "input": "", "output": "Not all chemotherapy drugs cause hair loss, and it varies depending on the specific drugs and doses used in your treatment plan. Hair loss is one of the possible side effects, but your oncologist can tell you what to expect with your specific regimen. If hair loss is a concern for you, some people also explore options like cold cap therapy, though it's worth discussing with your care team whether that's appropriate for your treatment."},
  {"instruction": "Will I lose my eyebrows and eyelashes too, not just hair on my head?", "input": "", "output": "It's possible \u2014 some chemotherapy regimens can affect hair all over the body, not just the scalp, including eyebrows and eyelashes, though this varies by drug and individual. This can be a genuinely hard adjustment, and it's okay to grieve these changes even while staying focused on treatment. Your care team may have resources, like makeup or styling tips, that other patients have found helpful."},
  {"instruction": "How can I manage nausea from chemotherapy?", "input": "", "output": "Nausea is one of the more common chemotherapy side effects, but your care team has anti-nausea medications and strategies specifically designed to help manage it, so please don't hesitate to bring it up if you're struggling. Everyone responds differently, and it sometimes takes some adjustment to find what works best for you. Let your oncology team know right away if your current approach isn't providing enough relief."},
  {"instruction": "Can I still work while going through breast cancer treatment?", "input": "", "output": "Many people do continue working during treatment, though it often depends on your specific treatment plan, side effects, and the nature of your job \u2014 some people need to reduce hours or take leave during more intensive phases like chemotherapy or surgery recovery. This is a very individual decision, and your care team, along with your employer's HR department, can help you understand your options, including medical leave protections. Please be gentle with yourself as you figure out what balance works for you."},
  {"instruction": "What is a clinical trial and should I consider joining one?", "input": "", "output": "A clinical trial is a research study testing new treatments, treatment combinations, or approaches to breast cancer care, and joining one is entirely optional but can be a way to access newer therapies not yet widely available. Whether it's right for you depends on your specific diagnosis, treatment history, and personal goals. Your doctor or NCI's Cancer Information Service can help you find and evaluate trials that might be a good fit."},
  {"instruction": "Are clinical trials safe?", "input": "", "output": "Clinical trials go through careful ethical review and monitoring processes designed to protect participants, though like any medical treatment, they do carry some uncertainty since researchers are still studying how well the treatment works. Your care team can walk you through the specific risks and potential benefits of any trial you're considering. It's completely reasonable to ask lots of questions before deciding whether a trial is right for you."},
  {"instruction": "What's the difference between luminal A and luminal B breast cancer?", "input": "", "output": "Both are hormone receptor-positive subtypes, but luminal A cancers tend to be HER2-negative with lower Ki-67 (a marker of how fast cells are dividing), generally making them slower-growing and associated with a better prognosis, while luminal B cancers tend to have higher Ki-67 and can be HER2-positive or negative, generally growing somewhat faster. Your doctor can explain which subtype applies to you and how it's shaping your treatment plan."},
  {"instruction": "What does Ki-67 score mean?", "input": "", "output": "Ki-67 is a marker that measures how quickly cancer cells are dividing \u2014 a higher Ki-67 score suggests the cancer is growing more quickly, while a lower score suggests slower growth. It's one of several pieces of information, alongside hormone receptor and HER2 status, that helps your doctor determine your cancer's molecular subtype and plan treatment. Ask your doctor what your specific Ki-67 result means for your case."},
  {"instruction": "What is HER2-low breast cancer and how is it different from HER2-positive?", "input": "", "output": "HER2-low means the cancer cells have a small amount of the HER2 protein on their surface \u2014 not enough to be classified as fully HER2-positive, but enough that certain targeted therapies, like trastuzumab deruxtecan, can still be effective. This is a relatively newer classification that has opened up additional treatment options for people who previously wouldn't have qualified for HER2-targeted therapy. Your oncologist can explain what your HER2-low status means for your treatment options."},
  {"instruction": "What is a survivorship care plan?", "input": "", "output": "A survivorship care plan is a document your doctor may give you that details the treatments you received and outlines a plan for your ongoing follow-up care, including which tests you'll need and how often. It's meant to help you and any future doctors understand your cancer history and stay on top of monitoring for recurrence or late effects. If you haven't received one, it's worth asking your care team for one as you finish active treatment."},
  {"instruction": "What tests will I have at my follow-up appointments after treatment?", "input": "", "output": "Common follow-up tests include a physical exam, clinical breast exam, mammogram, and sometimes breast MRI, neurological exam, or pelvic exam, depending on your specific treatment history. The exact schedule and tests will be outlined in your survivorship care plan. Your doctor can explain exactly what your personal follow-up schedule will include and why."},
  {"instruction": "Can I get emotional support during breast cancer treatment, not just medical care?", "input": "", "output": "Yes, and please do reach out for it \u2014 support groups, social workers on your care team, and mental health professionals who specialize in supporting people with cancer are all valuable resources. You don't have to manage the emotional weight of this diagnosis on your own. Ask your care team what specific support resources are available to you locally or virtually."},
  {"instruction": "How can I find a breast cancer support group?", "input": "", "output": "Your care team, particularly a social worker, can often point you toward local or virtual support groups specifically for people with breast cancer, and organizations like NCI's Cancer Information Service can also help connect you with resources. Some people find in-person groups most helpful, while others prefer online communities \u2014 there's no wrong choice. It's worth trying a few different options to see what feels like the right fit for you."},
  {"instruction": "Is it normal to feel disconnected from my body after breast surgery?", "input": "", "output": "Yes, this is a very common feeling after breast surgery, whether it's a lumpectomy, mastectomy, or reconstruction \u2014 changes to your body can understandably affect how connected you feel to it. These feelings are valid and don't need to be minimized or rushed through. Talking to a counselor or others who've been through similar surgery can help, and your care team can point you toward those resources if you'd like."},
  {"instruction": "What is a modified radical mastectomy?", "input": "", "output": "A modified radical mastectomy removes the entire breast along with some of the underarm lymph nodes, but unlike an older, more extensive version of this surgery, it preserves the chest wall muscles. It's sometimes used, including during pregnancy, when a full mastectomy is needed. Your surgeon can explain exactly what type of mastectomy is being recommended for you and why."},
  {"instruction": "How do doctors decide whether to remove lymph nodes during breast cancer surgery?", "input": "", "output": "Doctors typically start with a sentinel lymph node biopsy, checking the first lymph node(s) most likely to contain cancer cells if the cancer has spread beyond the breast. If cancer is found there, more lymph nodes may be removed either during the same surgery or in a follow-up procedure. Your surgeon can explain the specific approach planned for your case and how many nodes might be involved."},
  {"instruction": "What are the drugs used for triple-negative breast cancer specifically?", "input": "", "output": "Since triple-negative breast cancer lacks hormone receptors and HER2, it doesn't respond to hormone therapy or HER2-targeted drugs, so treatment typically relies on chemotherapy, sometimes combined with the immunotherapy drug pembrolizumab or the targeted therapy drug sacituzumab govitecan. Your oncologist will explain which specific combination is recommended based on your cancer's exact characteristics and stage."},
  {"instruction": "What targeted therapies exist for BRCA-related breast cancer?", "input": "", "output": "Olaparib and talazoparib are targeted therapy drugs, known as PARP inhibitors, that may be used to treat breast cancer linked to BRCA1 or BRCA2 mutations, since these drugs interfere with the cancer cells' ability to repair damaged DNA. Your oncologist can explain whether one of these is appropriate for your treatment plan based on your genetic testing results."},
  {"instruction": "How does inflammatory breast cancer get treated differently from other types?", "input": "", "output": "Inflammatory breast cancer often requires a combination approach involving chemotherapy first, followed by surgery and radiation, since it tends to be more aggressive and spread through the skin's lymph vessels. Because of this aggressive nature, treatment usually starts more urgently than with some other types. Your oncology team can explain the specific sequence and combination of treatments planned for your case."},
  {"instruction": "Why does my doctor keep mentioning 'restaging' after my cancer came back?", "input": "", "output": "Restaging means repeating many of the same tests you had at initial diagnosis, like imaging and lab work, to determine exactly where and how far the recurrence has spread. This new assessment helps your doctor plan the most appropriate treatment for this new phase, and your restaged result will usually have an 'r' added to reflect that it's a new assessment. It's a standard and important part of figuring out next steps after a recurrence."},
  {"instruction": "What kind of doctor should I see for breast cancer treatment?", "input": "", "output": "Your care team will likely include several specialists working together \u2014 commonly a breast surgeon, medical oncologist, radiation oncologist, and sometimes a plastic surgeon if reconstruction is involved, along with nurses, social workers, and genetic counselors as needed. This team-based approach means different experts are focused on different parts of your care. If you're just starting out, your primary care doctor or an initial oncologist referral is usually the first step."},
  {"instruction": "Can breast cancer treatment be done entirely as an outpatient?", "input": "", "output": "Many parts of breast cancer treatment, including chemotherapy, radiation, and many biopsies, are done on an outpatient basis, meaning you go home the same day. Surgery, particularly mastectomy, may sometimes require a short hospital stay depending on the extent of the procedure and your individual recovery needs. Your care team can tell you what to expect for each specific part of your treatment plan."},
  {"instruction": "How do I know if a lump I feel is something to be concerned about?", "input": "", "output": "It's hard to tell the difference between a benign and a concerning lump just by touch alone, which is exactly why any new lump deserves a doctor's evaluation rather than a guess. Most breast lumps turn out to be benign, but getting it checked gives you real answers instead of ongoing worry. Please don't try to self-diagnose \u2014 make an appointment and let a professional examine it properly."},
  {"instruction": "What's the difference between incisional and excisional biopsy?", "input": "", "output": "An incisional biopsy removes just part of a lump or a tissue sample, while an excisional biopsy removes the entire lump, sometimes along with a small margin of healthy surrounding tissue. Both are types of surgical biopsy and typically involve local or general anesthesia. Your surgeon can explain which type is being recommended for your specific situation and why."},
  {"instruction": "Why would my doctor recommend an image-guided biopsy?", "input": "", "output": "Image-guided biopsy, using ultrasound, mammography, or MRI to direct the needle, is often used when the abnormal area is too deep to feel or when the doctor can't locate a lump by touch alone. It helps ensure the sample is taken from precisely the right spot. Your doctor can explain which imaging method will guide your specific biopsy procedure."}
]

In [12]:
len(a)

173

In [10]:
134 + 83

217

In [26]:
len(content + a)

256

In [27]:
merged_dict = content + a

In [28]:
len(merged_dict)

256

In [33]:
merged_dict[250]

{'instruction': "Why does my doctor keep mentioning 'restaging' after my cancer came back?",
 'input': '',
 'output': "Restaging means repeating many of the same tests you had at initial diagnosis, like imaging and lab work, to determine exactly where and how far the recurrence has spread. This new assessment helps your doctor plan the most appropriate treatment for this new phase, and your restaged result will usually have an 'r' added to reflect that it's a new assessment. It's a standard and important part of figuring out next steps after a recurrence."}

In [34]:
merged_dataset = pd.read_csv("/content/merged_medical_qa_full.csv")

In [35]:
merged_dataset.shape

(14336, 2)

In [36]:
merged_dataset.head()

,question,answer
0,Who is at risk for Lymphocytic Choriomeningiti...,LCMV infections can occur after exposure to fr...
1,What are the symptoms of Lymphocytic Choriomen...,LCMV is most commonly recognized as causing ne...
2,How to diagnose Lymphocytic Choriomeningitis (...,"During the first phase of the disease, the mos..."
3,What are the treatments for Lymphocytic Chorio...,"Aseptic meningitis, encephalitis, or meningoen..."
4,How to prevent Lymphocytic Choriomeningitis (L...,LCMV infection can be prevented by avoiding co...


In [38]:
BREAST_CANCER_KEYWORDS = [
    "breast cancer", "breast tumor", "breast tumour", "mammogram", "mammography",
    "brca", "mastectomy", "lumpectomy", "breast biopsy", "breast lump",
    "ductal carcinoma", "lobular carcinoma", "her2", "triple negative",
    "breast self-exam", "breast screening",
]

In [39]:
pattern = "|".join(re.escape(k) for k in BREAST_CANCER_KEYWORDS)

In [41]:
mask = (
    merged_dataset["question"].str.contains(pattern, case=False, na=False)
    | merged_dataset["answer"].str.contains(pattern, case=False, na=False)
)

In [46]:
general_medData = merged_dataset[~mask].reset_index(drop=True)

In [47]:
general_medData.shape

(14253, 2)

In [49]:
general_medData = general_medData.sample(frac=1, random_state=42).reset_index(drop=True)

In [50]:
general_instruction_pairs = [
    {
        "instruction": row["question"],
        "input": "",
        "output": row["answer"],
    }
    for _, row in general_medData[:750].iterrows()
]

In [51]:
len(general_instruction_pairs)

750

In [53]:
general_instruction_pairs[0]

{'instruction': 'What is the outlook for Arachnoiditis ?',
 'input': '',
 'output': 'Arachnoiditis is adisorder that causes chronic pain and neurological deficits and does not improve significantly with treatment.Surgery may only provide temporary relief. The outlook for someone witharachnoiditis iscomplicated by the fact that the disorder has no predictable pattern or severity of symptoms.'}

In [54]:
merged_dict = merged_dict + general_instruction_pairs

In [56]:
len(merged_dict)

1006

In [57]:
merged_dict[1000]

{'instruction': 'What is (are) adenosine deaminase deficiency ?',
 'input': '',
 'output': 'Adenosine deaminase (ADA) deficiency is an inherited disorder that damages the immune system and causes severe combined immunodeficiency (SCID). People with SCID lack virtually all immune protection from bacteria, viruses, and fungi. They are prone to repeated and persistent infections that can be very serious or life-threatening. These infections are often caused by "opportunistic" organisms that ordinarily do not cause illness in people with a normal immune system.  The main symptoms of ADA deficiency are pneumonia, chronic diarrhea, and widespread skin rashes. Affected children also grow much more slowly than healthy children and some have developmental delay.  Most individuals with ADA deficiency are diagnosed with SCID in the first 6 months of life. Without treatment, these babies usually do not survive past age 2. In about 10 percent to 15 percent of cases, onset of immune deficiency is de

In [58]:
seen = set()
unique_merged_dict = []
for d in merged_dict:
    # Convert dictionary to a hashable representation (frozenset of items)
    # The order of key-value pairs in a dict doesn't matter for equality, so frozenset(d.items()) is a good choice.
    hashable_d = frozenset(d.items())
    if hashable_d not in seen:
        seen.add(hashable_d)
        unique_merged_dict.append(d)

merged_dict = unique_merged_dict

In [59]:
len(merged_dict)

1006

In [60]:
with open("breast_cancer_finetune_data.json", "w") as f:
    json.dump(merged_dict, f, indent=2)

# Safety tuning in LLM

In [65]:
from datasets import load_dataset

# 1. Load the dataset
medquad = load_dataset("lavita/MedQuAD", split="train")

# 2. Convert to a pandas DataFrame
safety_df = medquad.to_pandas()

In [66]:
safety_df.head()

,document_id,document_source,document_url,category,umls_cui,umls_semantic_types,umls_semantic_group,synonyms,question_id,question_focus,question_type,question,answer
0,0000559,GHR,https://ghr.nlm.nih.gov/condition/keratoderma-...,None,C0343073,T047,Disorders,KWWH,0000559-1,keratoderma with woolly hair,information,What is (are) keratoderma with woolly hair ?,Keratoderma with woolly hair is a group of rel...
1,0000559,GHR,https://ghr.nlm.nih.gov/condition/keratoderma-...,None,C0343073,T047,Disorders,KWWH,0000559-2,keratoderma with woolly hair,frequency,How many people are affected by keratoderma wi...,Keratoderma with woolly hair is rare; its prev...
2,0000559,GHR,https://ghr.nlm.nih.gov/condition/keratoderma-...,None,C0343073,T047,Disorders,KWWH,0000559-3,keratoderma with woolly hair,genetic changes,What are the genetic changes related to kerato...,"Mutations in the JUP, DSP, DSC2, and KANK2 gen..."
3,0000559,GHR,https://ghr.nlm.nih.gov/condition/keratoderma-...,None,C0343073,T047,Disorders,KWWH,0000559-4,keratoderma with woolly hair,inheritance,Is keratoderma with woolly hair inherited ?,Most cases of keratoderma with woolly hair hav...
4,0000559,GHR,https://ghr.nlm.nih.gov/condition/keratoderma-...,None,C0343073,T047,Disorders,KWWH,0000559-5,keratoderma with woolly hair,treatment,What are the treatments for keratoderma with w...,These resources address the diagnosis or manag...


In [68]:
safety_df.columns

Index(['document_id', 'document_source', 'document_url', 'category',
       'umls_cui', 'umls_semantic_types', 'umls_semantic_group', 'synonyms',
       'question_id', 'question_focus', 'question_type', 'question', 'answer'],
      dtype='object')

In [69]:
safety_df = safety_df[['question', 'answer']]

In [70]:
safety_df.head()

,question,answer
0,What is (are) keratoderma with woolly hair ?,Keratoderma with woolly hair is a group of rel...
1,How many people are affected by keratoderma wi...,Keratoderma with woolly hair is rare; its prev...
2,What are the genetic changes related to kerato...,"Mutations in the JUP, DSP, DSC2, and KANK2 gen..."
3,Is keratoderma with woolly hair inherited ?,Most cases of keratoderma with woolly hair hav...
4,What are the treatments for keratoderma with w...,These resources address the diagnosis or manag...


In [71]:
safety_df.shape

(47441, 2)

In [72]:
safety_instruction_pairs = [
    {
        "instruction": row["question"],
        "input": "",
        "output": row["answer"],
    }
    for _, row in safety_df[:300].iterrows()
]

In [73]:
merged_dict = merged_dict + safety_instruction_pairs

In [74]:
len(merged_dict)

1306

In [76]:
import random

random.shuffle(merged_dict)

In [80]:
merged_dict[1300]

{'instruction': 'How many people are affected by familial paroxysmal kinesigenic dyskinesia ?',
 'input': '',
 'output': 'Familial paroxysmal kinesigenic dyskinesia is estimated to occur in 1 in 150,000 individuals. For unknown reasons, this condition affects more males than females.'}

In [82]:
len(merged_dict)

1306

In [83]:
with open("breast_cancer_finetune_data2.json", "w") as f:
    json.dump(merged_dict, f, indent=2)